# Segmentar en Colab lo que el Mac no aguanta (visor3d)

Cuaderno de `tools/visor3d.py`. Corre TotalSegmentator en una GPU de Colab sobre un volumen que
**ya sale limpio del Mac** (`visor3d exporta-colab`): NIfTI sin cabeceras de paciente y, salvo
decisión expresa, sin la cabeza. Aquí nunca se sube un DICOM.

1. Entorno de ejecución → Cambiar tipo → **GPU**.
2. Primera vez: deja `MODO_ENSAYO = True`, que usa un TC público de ejemplo y no sube nada.
3. Uso real: `MODO_ENSAYO = False`, sube `volumen.nii.gz` y `manifiesto.json` de la carpeta que
   imprimió `exporta-colab`.
4. Descarga el zip, y en el Mac: `visor3d importa-colab --raiz … <serie> <zip>`.
5. Cuando la descarga haya terminado, corre la última celda: borra todo y suelta la máquina.

In [ ]:
MODO_ENSAYO = True          # True: TC público de ejemplo · False: tu volumen exportado
VERSION_ENSAYO = "2.14.0"   # la de TotalSegmentator en el Mac (solo para el ensayo)
import os, json, hashlib
os.makedirs("/content/trabajo", exist_ok=True)
os.chdir("/content/trabajo")

In [ ]:
def sha256(ruta):
    h = hashlib.sha256()
    with open(ruta, "rb") as f:
        for trozo in iter(lambda: f.read(1 << 20), b""):
            h.update(trozo)
    return h.hexdigest()

if MODO_ENSAYO:
    import urllib.request
    urllib.request.urlretrieve("https://raw.githubusercontent.com/wasserth/TotalSegmentator/"
                               "master/tests/reference_files/example_ct_sm.nii.gz", "volumen.nii.gz")
    manif = {"serie": "ensayo", "sha256_volumen": sha256("volumen.nii.gz"),
             "totalsegmentator": VERSION_ENSAYO,
             "tareas": [{"t": "total_completo", "tarea": "total", "kw": {"ml": True}}]}
else:
    from google.colab import files
    subidos = files.upload()
    faltan = {"volumen.nii.gz", "manifiesto.json"} - set(subidos)
    assert not faltan, "faltan ficheros (o Colab los renombró): %s" % sorted(faltan)
    manif = json.load(open("manifiesto.json"))
    assert sha256("volumen.nii.gz") == manif["sha256_volumen"], "el volumen no es el del manifiesto"
print("volumen OK · tareas:", [x["t"] for x in manif["tareas"]],
      "· TotalSegmentator", manif["totalsegmentator"])

In [ ]:
# La MISMA versión que el Mac: si no, el Mac rechaza las máscaras al importarlas.
import subprocess, sys
v = manif["totalsegmentator"]
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "TotalSegmentator==" + v])
assert r.returncode == 0, "no se pudo instalar TotalSegmentator %s: no se sigue con otra" % v
from totalsegmentator.config import setup_totalseg, set_config_key, get_config_key
setup_totalseg()
set_config_key("send_usage_stats", False)
assert get_config_key("send_usage_stats") is False, "la telemetría sigue encendida: no se sigue"
print("TotalSegmentator", v, "· telemetría apagada")

In [ ]:
import torch
assert torch.cuda.is_available(), "sin GPU: Entorno de ejecución → Cambiar tipo → GPU"
from totalsegmentator.python_api import totalsegmentator
for x in manif["tareas"]:
    print("segmentando", x["t"], "…", flush=True)
    totalsegmentator(input="volumen.nii.gz", output=x["t"] + ".nii.gz", task=x["tarea"],
                     device="gpu", quiet=True, **x["kw"])
print("hecho")

In [ ]:
import zipfile, importlib.metadata
json.dump({"totalsegmentator": importlib.metadata.version("totalsegmentator"),
           "sha256_volumen": manif["sha256_volumen"]}, open("version.json", "w"))
nombre_zip = "mascaras_%s.zip" % manif["serie"]
with zipfile.ZipFile(nombre_zip, "w", zipfile.ZIP_DEFLATED) as z:
    z.write("version.json")
    for x in manif["tareas"]:
        z.write(x["t"] + ".nii.gz")
print("zip:", nombre_zip, "%.0f MB" % (os.path.getsize(nombre_zip) / 1e6))
if not MODO_ENSAYO:
    from google.colab import files
    files.download(nombre_zip)

In [ ]:
# Solo cuando la descarga haya TERMINADO: borra todo y suelta la máquina.
import shutil
os.chdir("/content")
shutil.rmtree("/content/trabajo", ignore_errors=True)
if not MODO_ENSAYO:
    from google.colab import runtime
    runtime.unassign()
print("limpio")